In [9]:
import pandas as pd
import re
import csv
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --- 1. Load and Prepare YOUR Dataset ---
print("Loading your reddit_posts.csv file...")
try:
    # This configuration is tailored to handle the specific errors in your file
    df_full = pd.read_csv(
        '../data/reddit_posts.csv', 
        encoding='latin-1',
        on_bad_lines='skip',
        engine='python' # Using the more flexible python engine
    )
    # Immediately select ONLY the 'text' and 'class' columns to avoid issues
    df = df_full[['text', 'class']].copy()
    
    # Convert 'class' column to numerical labels
    df['label'] = df['class'].apply(lambda x: 1 if x == 'suicide' else 0)
    
    # Ensure all text is a string and drop any empty rows
    df['text'] = df['text'].astype(str)
    df = df[df['text'].str.strip().astype(bool)]
    
    print(f"Dataset loaded and prepared with {len(df)} rows.")

except Exception as e:
    print(f"An error occurred during data loading: {e}")
    df = pd.DataFrame({'text': [], 'label': []})


# --- 2. Fix Imbalance with Undersampling ---
print("\nHandling severe class imbalance with undersampling...")

df_majority = df[df.label == 0]
df_minority = df[df.label == 1]

# Undersample the majority class to be the same size as the minority class
df_majority_downsampled = df_majority.sample(n=len(df_minority), random_state=42)

# Create a new, perfectly balanced dataframe
df_balanced = pd.concat([df_majority_downsampled, df_minority])

# Shuffle the balanced dataframe
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nOriginal label distribution:\n{df['label'].value_counts(normalize=True)}")
print(f"\nNew BALANCED label distribution:\n{df_balanced['label'].value_counts(normalize=True)}")
print(f"Total rows in new balanced training dataset: {len(df_balanced)}")


# --- 3. Convert the BALANCED Dataset to Hugging Face Format ---
train_df, val_df = train_test_split(df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['label'])
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
print("\nConverted balanced dataset to Hugging Face format.")


# --- 4. Tokenize, Load Model (Standard Steps) ---
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


# --- 5. Set Up a Standard Trainer ---
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='binary', zero_division=0)
    acc = accuracy_score(p.label_ids, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

MODEL_OUTPUT_DIR = '../models/distilbert-finetuned-balanced'

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=3, # Train for 3 epochs on the balanced data
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch"
)

# Use the standard Trainer, as no custom logic is needed now
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


# --- 6. Train the Model on the Balanced Data ---
print("\nStarting to fine-tune the model on the BALANCED dataset...")
trainer.train()
print("Model fine-tuning complete.")


# --- 7. Save the Final, Working Model ---
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print(f"✅ Final balanced model and tokenizer saved to {MODEL_OUTPUT_DIR}")

Loading your reddit_posts.csv file...
Dataset loaded and prepared with 311 rows.

Handling severe class imbalance with undersampling...

Original label distribution:
label
0    0.514469
1    0.485531
Name: proportion, dtype: float64

New BALANCED label distribution:
label
1    0.5
0    0.5
Name: proportion, dtype: float64
Total rows in new balanced training dataset: 302

Converted balanced dataset to Hugging Face format.


Map:   0%|          | 0/241 [00:00<?, ? examples/s]

Map:   0%|          | 0/61 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting to fine-tune the model on the BALANCED dataset...


C:\Users\91952\Desktop\FINAL_PROJECT-1\mental-health-detection\backend\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.694700,0.671323,0.606557,0.707317,0.557692,0.966667
2,0.637400,0.590772,0.688525,0.753247,0.617021,0.966667
3,0.556600,0.396470,0.868852,0.866667,0.866667,0.866667


C:\Users\91952\Desktop\FINAL_PROJECT-1\mental-health-detection\backend\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\91952\Desktop\FINAL_PROJECT-1\mental-health-detection\backend\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\91952\Desktop\FINAL_PROJECT-1\mental-health-detection\backend\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Model fine-tuning complete.
✅ Final balanced model and tokenizer saved to ../models/distilbert-finetuned-balanced


In [8]:
print(df['label'].value_counts(normalize=True))


label
0    0.990672
1    0.009328
Name: proportion, dtype: float64
